In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from pathlib import Path


In [2]:
base_dir = Path("E:/Coronary_Heart_Disease_Detection/Data")

dataset_configs = {
    "Env1": {
        "csv": base_dir / "Generated" / "Env1" / "csv",
        "npy": base_dir / "Disease_dataset" / "Env1" / "NumpyData",
    },
    "Env2": {
        "csv": base_dir / "Generated" / "Env2" / "csv",
        "npy": base_dir / "Disease_dataset" / "Env2" / "NumpyData",
    },
    "Env3": {
        "csv": base_dir / "Generated" / "Env3" / "csv",
        "npy": base_dir / "Disease_dataset" / "Env3" / "NumpyData",
    },
    "Env4": {
        "csv": base_dir / "Generated" / "Env4" / "csv",
        "npy": base_dir / "Disease_dataset" / "Env4" / "NumpyData",
    },
    "Eval": {
        "csv": base_dir / "Generated" / "Eval" / "csv",
        "npy": base_dir / "Disease_dataset" / "Eval" / "NumpyData",
    },
}

for name, cfg in dataset_configs.items():
    cfg["npy"].mkdir(parents=True, exist_ok=True)
    if not cfg["csv"].exists():
        print(f"Warning: {name} CSV folder {cfg['csv']} not found")


In [3]:
def create_matrix(x_axis, y_axis, filename):
    # Example x and y values (replace with actual values)
    x_values = np.array(x_axis)  # replace with actual x-values
    y_values = np.array(y_axis)  # replace with actual y-values

    # CHANGE INPUT MATRIX SIZE HERE
    grid_size = 30
    # THIS SHOULD NOT BE CHANGE :( BECAUSE I HAVE TEST THE RANGE FOR ECG
    x_min, x_max = 400, 1400
    y_min, y_max = 400, 1400
    
    # Initialize the feature matrix (28x28 grid)
    feature_matrix = np.zeros((grid_size, grid_size), dtype=float)
    
    # Calculate the size of each cell in the grid
    x_step = (x_max - x_min) / grid_size
    y_step = (y_max - y_min) / grid_size
    
    # Populate the feature matrix based on x and y values
    for x, y in zip(x_values, y_values):
        if x_min <= x < x_max and y_min <= y < y_max:
            # Determine the cell index for x and y
            x_idx = int((x - x_min) / x_step)
            y_idx = int((y - y_min) / y_step)
            
            # Mark the cell as occupied
            feature_matrix[y_idx, x_idx] = 1

        # New row to insert (make sure it has the same number of columns)
    new_row = np.zeros((1, grid_size), dtype=float)
    
    # Step 3: Update the first value of the new row
    if "Resting" in filename:
        new_value = 0.1
    elif "Working" in filename:
        new_value = 0.5
    new_row[0, 0] = new_value  # Update the first value
    # Append the new row
    updated_array = np.append(feature_matrix, new_row, axis=0)
    return updated_array

In [4]:
for dataset_name, cfg in dataset_configs.items():
    csv_dir = cfg["csv"]
    npy_dir = cfg["npy"]

    if not csv_dir.exists():
        continue

    print(f"Processing {dataset_name} from {csv_dir}")

    for csv_file in sorted(csv_dir.glob("*.csv")):
        print(f"  -> {csv_file.name}")
        ecg_data = pd.read_csv(csv_file)
        ecg_data = ecg_data[ecg_data["Time"] <= 120]
        r_peaks = ecg_data[ecg_data['Peak'] == 3]
        rr_intervals = r_peaks['Time'].diff().dropna().reset_index(drop=True)
        rr_intervals_ms = rr_intervals * 1000
        rr_n_ms = rr_intervals_ms[:-1]
        rr_n1_ms = rr_intervals_ms[1:]

        input_data = create_matrix(rr_n_ms, rr_n1_ms, csv_file.name)
        output_path = npy_dir / f"{csv_file.stem}.npy"
        np.save(output_path, input_data)


Processing Env1 from E:\Coronary_Heart_Disease_Detection\Data\Generated\Env1\csv
  -> 2025-10-02T15_25_40.210068700Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_41.767953600Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_42.632590Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_43.352127Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_44.000365200Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_44.676686400Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_45.312977Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_45.939224Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_46.625107300Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_47.404434900Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_48.066521800Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_48.755353800Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_49.509863600Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_50.166586400Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_50.855787500Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_51.680009Z_0_Resting-Normal.csv
  -> 2025-10-02T15_25_

In [6]:
example_dir = dataset_configs["Env1"]["npy"]
example_files = sorted(example_dir.glob("*.npy"))

if example_files:
    example_path = example_files[0]
    print(f"Previewing: {example_path}")
    data = np.load(example_path)
    print(data)
else:
    print(f"No .npy files found in {example_dir}. Run the conversion cell first.")


Previewing: E:\Coronary_Heart_Disease_Detection\Data\Disease_dataset\Env1\NumpyData\2025-10-02T15_25_40.210068700Z_0_Resting-Normal.npy
[[0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0